# 🚘 License Plate Recognition - Qwen2-VL Fine-tuning with Unsloth (Google Colab)

Notebook này hỗ trợ huấn luyện (Fine-tune) mô hình VLM `unsloth/Qwen2-VL-2B-Instruct-bnb-4bit` để nhận diện ký tự trên biển số xe một cách chính xác.

### ⚠️ Quan trọng trước khi chạy:
1. Chọn **Runtime** -> **Change runtime type** -> chọn **T4 GPU** (hoặc GPU mạnh hơn).
2. Sau khi chạy xong, tải file `unsloth_finetune.zip` về máy, giải nén và đặt vào thư mục `files_model/unsloth_finetune` trên máy Mac của bạn.

### 📦 1. Cài đặt các thư viện cần thiết

In [1]:
# Cập nhật pip và cài đặt Unsloth phiên bản mới nhất
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers" "trl" peft accelerate bitsandbytes
!pip install transformers datasets pillow opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.9 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-tc_d_qr5/unsloth_f445d785dbf446078898e01e7c2eae8d
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-tc_d_qr5/unsloth_f445d785dbf446078898e01e7c2eae8d
  Resolved https://github.com/unslothai/unsloth.git to commit f62c26e63d28c218c579c29a87e96d32ccde2eae
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 33.3 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 54.1 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 53.9 MB/s  0:00

### ⚙️ 2. Khai báo các cấu hình & tập lệnh huấn luyện

In [2]:
import torch
from unsloth import FastVisionModel
from datasets import load_dataset
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from transformers import TextStreamer

MODEL_NAME = "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit"
OUTPUT_DIR = "outputs"
SEED = 3407
MAX_STEPS = 50

# Hướng dẫn OCR cho mô hình
INSTRUCTION = """
You are a world-class OCR expert specializing in recognizing all types of vehicle license plates
(cars, motorbikes, trucks, etc.) in any weather or lighting condition, including blurred, dirty,
or low-contrast images. Your recognition must be precise and avoid any confusion between
similar-looking characters (e.g., '0' and 'O', '1' and 'I', '8' and 'B').

Analyze the given image, which may contain one or multiple license plates.
For each license plate detected, extract and return ONLY its exact content,
using only the following valid characters: digits (0-9), uppercase letters (A-Z),
the hyphen (-), and the dot (.).

List each license plate you find on a separate line, with no extra words,
symbols, or explanations.
"""

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### 🤖 3. Tải mô hình Qwen2-VL-2B và thiết lập LoRA

In [3]:
# Tự động xác định dtype dựa trên GPU (Tesla T4 dùng float16, A100/L4 dùng bfloat16)
has_bf16 = torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if has_bf16 else torch.float16

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    dtype=dtype
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=SEED,
)

==((====))==  Unsloth 2026.6.9: Fast Qwen2_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

### 📊 4. Tải và xử lý tập dữ liệu

In [4]:
dataset = load_dataset("EZCon/taiwan-license-plate-recognition", split="train")
dataset = dataset.remove_columns(["xywhr", "is_electric_car"])
dataset = dataset.rename_column("license_number", "text")

def convert_to_conversation(sample):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": INSTRUCTION},
                    {"type": "image", "image": sample["image"]}
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": sample["text"]}
                ]
            }
        ]
    }

converted_dataset = [convert_to_conversation(sample) for sample in dataset]

README.md:   0%|          | 0.00/666 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

...

data/test-00018-of-00020.parquet:   0%|          | 0.00/5.36M [00:00<?, ?B/s]

data/test-00019-of-00020.parquet:   0%|          | 0.00/5.45M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1812 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/518 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/259 [00:00<?, ? examples/s]

### 🔍 5. Kiểm tra khả năng OCR của mô hình trước khi huấn luyện

In [5]:
FastVisionModel.for_inference(model)

test_image = dataset[2]["image"]
messages = [
    {"role": "user", "content": [
        {"type": "image", "image": test_image},
        {"type": "text", "text": INSTRUCTION}
    ]}
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(test_image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

print("\n--- KẾT QUẢ TRƯỚC KHI HUẤN LUYỆN ---")
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
result = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                       use_cache = True, temperature = 1.5, min_p = 0.1)


--- KẾT QUẢ TRƯỚC KHI HUẤN LUYỆN ---


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-

MYK-8008<|im_end|>


### 🏋️‍♂️ 6. Tiến hành huấn luyện (Fine-tuning)

In [6]:
FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=converted_dataset,
    args=SFTConfig(
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        output_dir=OUTPUT_DIR,
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048,
    ),
)

trainer_stats = trainer.train()

Unsloth: Model does not have a default image size - using 512


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,812 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 28,950,528 of 2,237,936,128 (1.29% trained)


Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bits

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bits

Step,Training Loss
1,2.578646
2,2.568707
3,2.553122
4,2.502776
5,2.348458
6,2.157708
7,1.920252
8,1.650316
9,1.410446
10,1.193429


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-50/tokenizer_config.json.


### 🎯 7. Kiểm tra khả năng OCR của mô hình sau khi huấn luyện

In [7]:
FastVisionModel.for_inference(model)

print("\n--- KẾT QUẢ SAU KHI HUẤN LUYỆN ---")
result = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                       use_cache = True, temperature = 1.5, min_p = 0.1)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- KẾT QUẢ SAU KHI HUẤN LUYỆN ---


/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bits

MYK-8008<|im_end|>


### 💾 8. Hợp nhất trọng số LoRA và lưu dưới dạng 16-bit (Tối ưu cho Mac M1)

In [8]:
# Lưu mô hình ở dạng merged 16bit để dễ dàng load bằng transformers thông thường trên macOS
model.save_pretrained_merged("unsloth_finetune", tokenizer, save_method="merged_16bit")
print("Mô hình đã được lưu thành công tại thư mục 'unsloth_finetune'.")

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in unsloth_finetune/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/4.42G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [03:23<00:00, 203.45s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:51<00:00, 111.37s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_finetune`
Mô hình đã được lưu thành công tại thư mục 'unsloth_finetune'.


### 📦 9. Đóng gói mô hình thành file .zip để tải về

In [9]:
!zip -r unsloth_finetune.zip unsloth_finetune
print("Đóng gói thành công file 'unsloth_finetune.zip'! Bạn có thể click chuột phải vào file này trong thanh sidebar của Colab để tải xuống.")

  adding: unsloth_finetune/ (stored 0%)
  adding: unsloth_finetune/chat_template.jinja (deflated 65%)
  adding: unsloth_finetune/.cache/ (stored 0%)
  adding: unsloth_finetune/.cache/huggingface/ (stored 0%)
  adding: unsloth_finetune/.cache/huggingface/.gitignore (stored 0%)
  adding: unsloth_finetune/.cache/huggingface/CACHEDIR.TAG (deflated 24%)
  adding: unsloth_finetune/.cache/huggingface/download/ (stored 0%)
  adding: unsloth_finetune/.cache/huggingface/download/model.safetensors.metadata (deflated 29%)
  adding: unsloth_finetune/config.json (deflated 76%)
  adding: unsloth_finetune/tokenizer.json (deflated 81%)
  adding: unsloth_finetune/generation_config.json (deflated 35%)
  adding: unsloth_finetune/model.safetensors (deflated 21%)
  adding: unsloth_finetune/processor_config.json (deflated 72%)
  adding: unsloth_finetune/tokenizer_config.json (deflated 81%)
Đóng gói thành công file 'unsloth_finetune.zip'! Bạn có thể click chuột phải vào file này trong thanh sidebar của Colab 

### 📥 10. Tự động tải file zip về máy local

In [13]:
# Chạy cell này để tự động kích hoạt tính năng tải file zip về trình duyệt
from google.colab import files
files.download('unsloth_finetune.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>